## 重要度

回帰モデルの説明変数の回帰性能に対する何らかの寄与を「重要度」と呼ぶことがあります。
通常は一つの回帰モデルの重要度を評価します。

### permutation importance

回帰スコアを利用する手法があります。
この手法はある説明変数を並び替える、もしくはランダム化した場合の回帰スコアの減少具合からその説明変数の重要性を直接評価する手法です。
回帰モデルを作成してから並び替えるのが標準のやり方ですが、並び替えてから回帰モデルを作成するなどの変種が考えられます。この手法は回帰スコアが計算できれば良いので、どの回帰モデルに対しても適用可能です。
この手法の欠点は説明変数を並び替える際に乱数の影響を受けることです。

教師あり学習の性能評価は教師データに対する一致度しかありませんから、
教師あり学習ではこれに類する手法しか本来はありません。

- 使用法
```python
from sklearn.inspection import permutation_importance
feature_importance = permutation_importance(reg, X, y, n_repeats=30, random_state=20)
```

### 線形回帰モデルの係数

教師あり学習の係数は教師データに対する一致度に直接影響を及ぼすパラメタです。
feature_importanceと異なり、相関が正なのか負なのかが同時に分かります。


### ランダムフォレストでの不純物による説明変数重要度

scikit-learnでの決定木回帰の説明変数重要度は各ノードに対して以下のように定義されます。
```
N_t / N * (impurity - N_t_R / N_t * right_impurity
                    - N_t_L / N_t * left_impurity)
```
回帰の場合のimpurityは標準でMSEです。

あるノードがある説明変数で分割されるとします。
N_tはあるノードのサンプル数、impurityはあるノードのimpurityです。
N_t_R、right_impurityは分解した右側のサンプル数とimpurityです。
N_t_L、left_impurityは分解した左側のサンプル数とimpurityです。
Nは全サンプルです。
下のノードへ行くほどノードに含まれるサンプル数N_tが少ないのでN_t / N は小さくなる重みがつけられています。
あるノードをある説明変数で分割した場合にその説明変数の重要度を上式で定義します。

ノードに対して重要度を定義したので、決定木に対しての重要度はノードの重要度の和で定義します。
random forest回帰では多数の決定木回帰の平均で予測値を得ます。
同様に、多数の決定木に対しての説明変数重要度を足し合わせ、
最後に全部の説明変数の重要度が１となるように規格化します。
簡単には決定木の上部にある説明変数の重要性が高くなります。

この定義の問題点はランダム変数の重要度が高いと誤った評価をされる場合があることです。
また、決定木を用いた回帰にしか用いることができません。
利点は回帰モデルを得ると説明変数の重要性が一意に決まることです。

- 使用法
```python
reg = RandomForestRegressor(n_estimators=100, random_state=1)
rf.feature_importances_
```

参考文献

1. https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html#sklearn.tree.DecisionTreeRegressor

2. Beware Default Random Forest Importances
Brought to you by explained.ai
Terence Parr, Kerem Turgutlu, Christopher Csiszar, and Jeremy Howard
March 26, 2018.
https://explained.ai/rf-importance/

